In [0]:
%run ./config

In [0]:
import json
from datetime import datetime
from dbruntime.databricks_repl_context import get_context
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StructType

In [0]:

dict_result = get_context().__dict__
_pipeline_name = str(dict_result.get("jobId", "interactive"))
_pipeline_run_id = str(
    dict_result.get("multitaskParentRunId")
    or dict_result.get("idInJob", "0")
)
_task_run_id = str(dict_result.get("idInJob", "0"))
_notebook_path = str(dict_result.get("notebookPath") or "unknown")

print(f"Pipeline Name : {_pipeline_name}")
print(f"Pipeline RunId: {_pipeline_run_id}")
print(f"Task RunId    : {_task_run_id}")
print(f"Notebook Path : {_notebook_path}")


In [0]:


# ===================================================================== #
# Audit helper - build and insert records into generic_job_audit        #
# Shared audit table with batch framework (same schema)                 #
# ===================================================================== #


def build_audit_record(
    batch_ctrl_key,
    source_name,
    interface_name,
    records_loaded,
    job_status,
    start_time,
    end_time,
    framework_name=None,
    error_message=None,
    custom_attributes=None,
    NOTEBOOK_PATH=None,
    extra_job_stats=None,
):
    """Build a dict representing one row in generic_job_audit."""
    elapsed = (
        (end_time - start_time).total_seconds()
        if start_time and end_time
        else 0.0
    )
    job_stats = {
        "records_loaded": records_loaded,
        "framework": framework_name or "streaming_ingestion_v2",
    }
    if extra_job_stats:
        job_stats.update(extra_job_stats)

    record = {
        "BATCH_CTRL_KEY": batch_ctrl_key,
        "JOB_ID": _pipeline_name,
        "CLUSTER_ID": dict_result.get("clusterId", ""),
        "ATTRIBUTE_NAME_1": "source_name",
        "ATTRIBUTE_VALUE_1": source_name or "",
        "ATTRIBUTE_NAME_2": "interface_name",
        "ATTRIBUTE_VALUE_2": interface_name or "",
        "ATTRIBUTE_NAME_3": "records_loaded",
        "ATTRIBUTE_VALUE_3": str(records_loaded),
        "ATTRIBUTE_NAME_4": "framework",
        "ATTRIBUTE_VALUE_4": framework_name or "streaming_ingestion_v2",
        "ATTRIBUTE_NAME_5": "error",
        "ATTRIBUTE_VALUE_5": (error_message or "")[:4000],
        "JOB_STATUS": job_status,
        "JOB_START_TIMESTAMP": start_time,
        "JOB_END_TIMESTAMP": end_time,
        "ELAPSED_TIME": elapsed,
        "JOB_STATS": json.dumps(job_stats),
        "NOTEBOOK_PATH": NOTEBOOK_PATH or _notebook_path,
    }
    if custom_attributes:
        record.update(custom_attributes)
    return record


def insert_audit_record(audit_record, audit_table):
    """Insert a single audit record into generic_job_audit.

    Uses SQL INSERT to avoid small-file overhead from createDataFrame +
    saveAsTable in continuous job mode.
    """
    try:
        # Build column list and Value expressions
        cols = []
        vals = []
        for k, v in audit_record.items():
            if v is None:
                continue
            cols.append(k)
            if isinstance(v, (int, float)):
                vals.append(str(v))
            elif isinstance(v, datetime):
                vals.append(
                    f"TIMESTAMP '{v.strftime('%Y-%m-%d %H:%M:%S.%f')}'"
                )  # f is added as it was requested
            else:
                escaped = str(v).replace("'", "''")
                vals.append(f"'{escaped}'")

        col_str = ", ".join(cols)
        val_str = ", ".join(vals)
        spark.sql(
            f"INSERT INTO {audit_table} ({col_str}) VALUES ({val_str})"
        )
        print(
            f"[AUDIT] Record inserted into {audit_table} ({audit_record.get('JOB_STATUS', '')})"
        )
    except Exception as e:
        print(f"[AUDIT] WARNING: Failed to insert audit record: {e}")


print("\u2705 Audit utility functions loaded")


In [0]:

# ===================================================================== #
# XML Flattening Helpers                                                #
# Ported from Batch LANDING_BRONZE + 03_bronze_to_silver                #
# Independent copy - no dependency on batch UTILS                       #
# ===================================================================== #


def flatten_structs_only(df, separator=""):
    """Flatten only StructType columns (not ArrayType).

    """

    def _has_struct_fields(schema):
        return any(isinstance(f.dataType, StructType) for f in schema.fields)

    while _has_struct_fields(df.schema):
        for field in df.schema.fields:
            col_name = field.name
            if isinstance(field.dataType, StructType):
                expanded = [
                    F.col(f"{col_name}.{sub.name}").alias(
                        f"{col_name}{separator}{sub.name}"
                    )
                    for sub in field.dataType.fields
                ]
                df = df.select(
                    *[
                        F.col(f"{c.name}")
                        for c in df.schema.fields
                        if c.name != col_name
                    ],
                    *expanded,
                )
                break
    return df


def flatten_nested_df(df, separator=""):
    """Recursively flatten all StructType and ArrayType columns.

    Structs are expanded via dot notation; arrays are exploded. Continues until
    the schema contains only scalar types.
    """

    def _needs_flattening(schema):
        return any(
            isinstance(f.dataType, (StructType, ArrayType))
            for f in schema.fields
        )

    while _needs_flattening(df.schema):
        new_cols = []
        for field in df.schema.fields:
            col_name = field.name
            if isinstance(field.dataType, StructType):
                for sub in field.dataType.fields:
                    new_cols.append(
                        F.col(f"{col_name}.{sub.name}").alias(
                            f"{sub.name}"
                        )
                        # Unprefixed: downstream custom notebooks rely on flat names
                        # NOTE: prefixed alias f"{col_name}{separator}{sub.name}" prevents
                        # column collisions but requires downstream column name updates.
                    )
            elif isinstance(field.dataType, ArrayType):
                df = df.withColumn(
                    col_name, F.explode_outer(F.col(f"{col_name}"))
                )
                new_cols.append(F.col(f"{col_name}"))
            else:
                new_cols.append(F.col(f"{col_name}"))

        if new_cols:
            df = df.select(*new_cols)
    return df


def _pivot_xml_name_value_pairs(df, name_col="Name", value_col="Value"):
    """Pivot name-value pair arrays into columns.

    Handles XML patterns where data is stored as:
    <field><Name>key</Name><Value>val</Value></field>
    """
    if name_col not in df.columns or value_col not in df.columns:
        return df

    # SC-safe: no .rdd usage; .limit(500) caps driver memory
    distinct_names = [
        row[0]
        for row in df.select(name_col)
        .filter(F.col(name_col).isNotNull())
        .distinct()
        .limit(500)
        .collect()
    ]

    if not distinct_names:
        return df

    group_cols = [c for c in df.columns if c not in (name_col, value_col)]
    pivot_df = df.groupBy(group_cols).pivot(name_col, distinct_names).agg(
        F.first(value_col)
    )

    return pivot_df

In [0]:



def cast_columns_optimized(df, cast_dict, safe=False):
    """Cast DataFrame columns to specified types in a single .select().

    For each key in cast_dict, if the column exists in df it is cast
    to the target type; if it does not exist, a NULL literal of that
    type is added.  When safe=True, uses try_cast() instead of cast()
    so unparseable values become NULL instead of raising errors.

    Parameters
    ----------
    df : pyspark.sql.DataFrame
    cast_dict : dict
        {column_name: target_type_string}  e.g. {"qty": "int", "amt": "decimal(18,4)"}
    safe : bool, default False
        When True, uses try_cast (returns NULL on failure instead of error).

    Returns
    -------
    pyspark.sql.DataFrame
    """
    existing_cols = set(df.columns)
    select_exprs = []
    for col_name, target_type in cast_dict.items():
        if col_name in existing_cols:
            if safe:
                select_exprs.append(
                    F.expr(f"try_cast({col_name} AS {target_type})").alias(
                        col_name
                    )
                )
            else:
                select_exprs.append(
                    F.col(f"{col_name}")
                    .cast(target_type)
                    .alias(col_name)
                )
        else:
            select_exprs.append(
                F.lit(None).cast(target_type).alias(col_name)
            )
    return df.select(*select_exprs)


print("\u2705 cast_columns_optimized loaded")





In [0]:


def write_to_target_table(
    df,
    target_table,
    batch_control_key,
    partition_column=None,
    mode="append",
    merge_schema=True,
    add_audit_columns=True,
    audit_ts_column="_insert_timestamp",
    audit_batch_column="_insert_batch_id",
):
    """Write a DataFrame to a Delta target table with standard audit columns.

    This is the single write method for both silver and gold layers.
    Custom notebooks should call this after their transformation logic
    instead of writing Delta directly.

    Parameters
    ----------
    df : pyspark.sql.DataFrame
        The final transformed DataFrame to write.
    target_table : str
        Fully qualified Delta table name (catalog.schema.table).
    batch_control_key : str
        Pipeline identifier - used in the audit batch_id column.
    partition_column : str, optional
        Column to partition by. Ignored if column not present in df.
    mode : str, default 'append'
        Spark write mode: 'append', 'overwrite'.
    merge_schema : bool, default True
        Enable Delta mergeSchema on write.
    add_audit_columns : bool, default True
        When True, adds _insert_timestamp and _insert_batch_id columns.
    audit_ts_column : str, default '_insert_timestamp'
        Name of the audit timestamp column.
    audit_batch_column : str, default '_insert_batch_id'
        Name of the audit batch ID column.

    Returns
    -------
    int
        Number of records written.
    """
    # Quick empty check - avoids full count() scan (OOM-safe)
    if df.limit(1).count() == 0:
        print(
            f"[write_to_target_table] 0 records - skipping write to {target_table}"
        )
        return 0

    # Add standard audit columns
    output_df = df
    if add_audit_columns:
        batch_id = f"{batch_control_key}{datetime.now().strftime('%Y%m%d%H%M%S')}"
        output_df = output_df.withColumn(
            audit_ts_column, F.current_timestamp()
        ).withColumn(audit_batch_column, F.lit(batch_id))

    # Build writer
    writer = output_df.write.format("delta").mode(mode)

    if merge_schema:
        writer = writer.option("mergeSchema", "true")

    # Apply partitioning only if column exists in the DataFrame
    if (
        partition_column
        and partition_column.strip()
        and partition_column in output_df.columns
    ):
        writer = writer.partitionBy(partition_column)

    writer.saveAsTable(target_table)

    # Get actual count from Delta commit Log - avoids a second full scan
    try:
        last_op = (
            spark.sql(f"DESCRIBE HISTORY {target_table} LIMIT 1")
            .select("operationMetrics")
            .collect()[0][0]
        )
        record_count = int(last_op.get("numOutputRows", 0))
    except Exception:
        record_count = -1  # Unknown but write succeeded

    print(
        f"[write_to_target_table] {record_count} records written to {target_table} (mode='{mode}')"
    )
    return record_count


print("\u2705 write_to_target_table loaded")


In [0]:

def validate_silver_input(
    df, business_columns, required_key_columns, interface_name=""
):
    """Validate framework output for a custom silver notebook.

    Four-stage guard that protects Silver from upstream parsing failures:
    1. Required key columns must be present in the DataFrame.
    2. Missing optional business columns are added as NULL.
    3. At least one business value must be non-NULL across all rows.
    4. Key columns must not contain NULL values in any row.

    Parameters
    ----------
    df : pyspark.sql.DataFrame
        Framework output from the global temp view.
    business_columns : list[str]
        All expected business columns (required + optional).
    required_key_columns : list[str]
        Subset of business_columns that must be present and non-NULL.
        Used to build _record_guid.
    interface_name : str
        Name for error messages (e.g. 'INVENTORY_ADJUSTMENTS').

    Returns
    -------
    pyspark.sql.DataFrame
        Input DataFrame with any missing optional columns added as NULL.

    Raises
    ------
    ValueError
        If required columns are missing, all business values are NULL,
        or any key column contains NULL.
    """
    # Stage 1 - required columns must exist
    missing_required = [
        c for c in required_key_columns if c not in df.columns
    ]
    if missing_required:
        raise ValueError(
            f"Framework output is missing required {interface_name} columns: "
            + ", ".join(missing_required)
            + ". Check the XSD/root structure and framework XML parsing before writing Silver."
        )

    # Stage 2 - add missing optional columns as NULL
    missing_business = [c for c in business_columns if c not in df.columns]
    if missing_business:
        df = df.withColumns(
            {c: F.lit(None) for c in missing_business}
        )

    # Stage 3 - at least one business value must be non-NULL
    any_business_value = functools.reduce(
        lambda left, right: left | right,
        [F.col(c).isNotNull() for c in business_columns],
    )
    if df.filter(any_business_value).limit(1).count() == 0:
        raise ValueError(
            f"Framework output contains NULL for every {interface_name} business field. "
            "The failure is upstream of this custom notebook, most likely XML/XSD parsing. "
            "Silver write has been stopped."
        )

    # Stage 4 - key columns must not contain NULL (prevents unstable GUIDs)
    null_key_condition = None
    for c in required_key_columns:
        cond = F.col(c).isNull()
        null_key_condition = (
            cond
            if null_key_condition is None
            else (null_key_condition | cond)
        )

    if df.filter(null_key_condition).limit(1).count() > 0:
        raise ValueError(
            "One or more records have NULL Silver key fields: "
            + ", ".join(required_key_columns)
            + ". Correct the source/XSD parsing rather than generating an incomplete GUID."
        )

    return df


print("\u2705 validate_silver_input loaded")



In [0]:

# Single source of truth for framework metadata columns carried
# from bronze through to silver. Every custom silver notebook that
# uses rename_with_metadata() inherits this list automatically.

FRAMEWORK_METADATA_COLUMNS = [
    "_source_system_code",
    "_division_code",
    "_data_domain",
    "_pipeline_name",
    "_pipeline_run_id",
    "_created_ts",
    "_updated_ts",
    "partition_date",
]


def rename_with_metadata(df, rename_map):
    """Rename camelCase business columns to snake_case and carry framework metadata.

    Selects only the columns listed in rename_map (renamed) plus any
    FRAMEWORK_METADATA_COLUMNS present in df. This guarantees that the
    Silver DataFrame contains exactly the agreed business fields and
    framework audit columns - nothing more, nothing less.

    Parameters
    ----------
    df : pyspark.sql.DataFrame
        Validated framework output (after validate_silver_input).
    rename_map : dict[str, str]
        {source_camelCase: target_snake_case} - interface-specific.

    Returns
    -------
    pyspark.sql.DataFrame
        DataFrame with renamed business columns + framework metadata.
    """
    select_exprs = [
        F.col(src).alias(dst) for src, dst in rename_map.items()
    ]
    select_exprs += [
        F.col(c) for c in FRAMEWORK_METADATA_COLUMNS if c in df.columns
    ]
    return df.select(*select_exprs)



In [0]:



def create_record_guid(df, key_exprs, column_name="_record_guid"):
    """Create an MD5 hash column from concatenated key expressions.

    Parameters
    ----------
    df : pyspark.sql.DataFrame
    key_exprs : list[Column]
        Column expressions to concatenate and hash.
    column_name : str, default '_record_guid'
        Name of the output hash column. Override to reuse this
        method for other hash fields (e.g. '_dedup_key', '_match_key').

    Returns
    -------
    pyspark.sql.DataFrame
        DataFrame with the hash column as the first column.
    """
    existing_cols = df.columns
    df = df.withColumn(
        column_name, F.md5(F.concat_ws("~", *key_exprs))
    )
    df = df.select(
        column_name, *[c for c in existing_cols if c != column_name]
    )
    print(
        f"Silver final schema: {len(existing_cols) + 1} columns"
    )
    return df


print(
    "\u2705 FRAMEWORK_METADATA_COLUMNS, rename_with_metadata, create_record_guid loaded"
)



In [0]:


def write_to_silver(
    df,
    silver_table_name,
    batch_control_key,
    partition_column=None,
    add_audit_columns=False,
):
    """Schema-align, INSERT (append), and reconcile a silver DataFrame.

    If the target table exists, aligns DataFrame column types to the
    target schema before writing. Uses write_to_target_table() for
    the actual append.

    Parameters
    ----------
    df : pyspark.sql.DataFrame
        Fully transformed silver DataFrame ready to write.
    silver_table_name : str
        Fully qualified Delta table name (catalog.schema.table).
    batch_control_key : str
        Pipeline identifier (used by write_to_target_table).
    partition_column : str, optional
        Column to partition by.
    add_audit_columns : bool, default False
        Whether write_to_target_table should add _insert_timestamp / _insert_batch_id.

    Returns
    -------
    int
        Number of records written.
    """
    # --- Schema alignment against target table ---
    # Cache columns and schema once to avoid repeated Analysis RPCs (SEPAP001)
    existing_cols = df.columns
    existing_schema = df.schema
    try:
        tgt_fields = spark.table(silver_table_name).schema.fields
        tgt_schema = {f.name: f.dataType for f in tgt_fields}

        cast_exprs = {
            c: F.col(c).cast(tgt_schema[c])
            for c in existing_cols
            if c in tgt_schema and existing_schema[c].dataType != tgt_schema[c]
        }
        if cast_exprs:
            df = df.withColumns(cast_exprs)
            print(
                f"\u2713 Aligned {len(cast_exprs)} column type(s) to target table"
            )
    except AnalysisException:
        pass  # Table does not exist yet; first write will create it

    # --- Input count for reconciliation ---
    input_count = df.count()

    # --- INSERT (append) via common writer ---
    records_processed = write_to_target_table(
        df,
        silver_table_name,
        batch_control_key,
        partition_column=partition_column,
        add_audit_columns=add_audit_columns,
    )

    # --- Reconciliation ---
    if records_processed >= 0 and records_processed != input_count:
        print(
            f"\u26a0 Reconciliation warning: input={input_count}, written={records_processed}"
        )
    else:
        print(
            f"\u2713 Reconciliation passed: {input_count} records"
        )

    return records_processed


print("\u2705 write_to_silver loaded")